# 03 - Train YOLOv8 (Baseline)

This notebook trains and evaluates two YOLOv8 variants on the license plate detection dataset:
- **YOLOv8n** (nano) — fastest, smallest
- **YOLOv8s** (small) — better accuracy, still fast

| Step | Description |
|---|---|
| 1 | Install & import `ultralytics` |
| 2 | Detect device (MPS / CUDA / CPU) |
| 3 | Train YOLOv8n for 100 epochs |
| 4 | Validate — print mAP@50, mAP@50:95, precision, recall |
| 5 | Visualise predictions on test images |
| 6 | Train YOLOv8s and compare |
| 7 | Copy best weights to `models/` |

## 1. Install & Import

In [1]:
# On Google Colab run this cell; locally skip or install once via pip
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %pip install -q ultralytics

In [2]:
import os
import glob
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
from ultralytics import YOLO

print(f'PyTorch  : {torch.__version__}')
try:
    import ultralytics
    print(f'Ultralytics: {ultralytics.__version__}')
except Exception:
    pass

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/alijaafar/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch  : 2.8.0
Ultralytics: 8.4.27


## 2. Device Detection

In [3]:
def get_device():
    """Return the best available device string for ultralytics."""
    if torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return '0'   # first CUDA GPU
    return 'cpu'

DEVICE = get_device()
print(f'Using device: {DEVICE}')

if DEVICE == 'mps':
    print('  Apple Silicon MPS detected — GPU acceleration enabled.')
elif DEVICE == '0':
    print(f'  CUDA GPU: {torch.cuda.get_device_name(0)}')
else:
    print('  No GPU found — training on CPU (will be slow).')

Using device: mps
  Apple Silicon MPS detected — GPU acceleration enabled.


## 3. Configuration & Paths

In [4]:
# All paths are relative to this notebook's location (notebooks/)
NOTEBOOK_DIR = Path(os.getcwd())               # .../license-plate-detection/notebooks
PROJECT_DIR  = NOTEBOOK_DIR.parent             # .../license-plate-detection

if IN_COLAB:
    # Adjust to your Drive path
    PROJECT_DIR = Path('/content/drive/MyDrive/license-plate-detection')

DATA_YAML   = PROJECT_DIR / 'data' / 'processed' / 'yolo' / 'dataset.yaml'
TEST_IMG_DIR = PROJECT_DIR / 'data' / 'processed' / 'yolo' / 'images' / 'test'
RESULTS_DIR = PROJECT_DIR / 'results' / 'yolo'
MODELS_DIR  = PROJECT_DIR / 'models'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
EPOCHS    = 100
BATCH     = 16
IMG_SIZE  = 640
PATIENCE  = 20    # early stopping patience

print('Paths')
print(f'  dataset yaml : {DATA_YAML}')
print(f'  test images  : {TEST_IMG_DIR}')
print(f'  results      : {RESULTS_DIR}')
print(f'  models       : {MODELS_DIR}')
print()
print(f'Hyperparameters: epochs={EPOCHS}, batch={BATCH}, imgsz={IMG_SIZE}, patience={PATIENCE}')

assert DATA_YAML.exists(), f'dataset.yaml not found: {DATA_YAML}\nRun 02_data_preprocessing.ipynb first.'

Paths
  dataset yaml : /Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/dataset.yaml
  test images  : /Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/images/test
  results      : /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo
  models       : /Users/alijaafar/Documents/MLP/license-plate-detection/models

Hyperparameters: epochs=100, batch=16, imgsz=640, patience=20


## 4. Train YOLOv8n (Nano)

YOLOv8n is the **fastest and smallest** variant — good baseline.  
Pretrained on COCO; we fine-tune on our license plate data.

In [5]:
# Load pretrained YOLOv8n
model_nano = YOLO('yolov8n.pt')
print(model_nano.info())

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
(129, 3157200, 0, 8.8575488)


In [6]:
results_nano = model_nano.train(
    data      = str(DATA_YAML),
    epochs    = EPOCHS,
    batch     = BATCH,
    imgsz     = IMG_SIZE,
    device    = DEVICE,
    patience  = PATIENCE,       # stop early if no improvement
    project   = str(RESULTS_DIR),
    name      = 'yolov8n',
    exist_ok  = True,
    # Augmentation (ultralytics built-in)
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.0,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.0,
    # Logging
    plots     = True,
    save      = True,
    verbose   = True,
)

print('\nYOLOv8n training complete.')
print(f'Results saved to: {RESULTS_DIR / "yolov8n"}')

/Users/alijaafar/Documents/MLP/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


New https://pypi.org/project/ultralytics/8.4.28 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.27 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, 

## 5. Validate YOLOv8n — Metrics

In [7]:
# Load best checkpoint and validate on val split
best_nano = RESULTS_DIR / 'yolov8n' / 'weights' / 'best.pt'
model_nano_best = YOLO(str(best_nano))

val_nano = model_nano_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'val',
)

print('\n' + '=' * 50)
print('YOLOv8n — Validation Metrics')
print('=' * 50)
print(f'  mAP@50      : {val_nano.box.map50:.4f}')
print(f'  mAP@50:95   : {val_nano.box.map:.4f}')
print(f'  Precision   : {val_nano.box.mp:.4f}')
print(f'  Recall      : {val_nano.box.mr:.4f}')

Ultralytics 8.4.27 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M4)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5113.6±1546.8 MB/s, size: 446.8 KB)
val: Scanning /Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/labels/val.cache... 64 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 22.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.5s/it 10.0s.8sss
                   all         64         67      0.978      0.925      0.981        0.6
Speed: 0.3ms preprocess, 122.4ms inference, 0.0ms loss, 12.9ms postprocess per image
Results saved to /Users/alijaafar/Documents/MLP/license-plate-detection/notebooks/runs/detect/val

YOLOv8n — Validation Metrics
  mAP@50      : 0.9806
  mAP@50:95   : 0.6003
  Precision   : 0.9785
  Recall      : 0.9254


In [8]:
# Also evaluate on the held-out TEST split
test_nano = model_nano_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'test',
)

print('\n' + '=' * 50)
print('YOLOv8n — Test Split Metrics')
print('=' * 50)
print(f'  mAP@50      : {test_nano.box.map50:.4f}')
print(f'  mAP@50:95   : {test_nano.box.map:.4f}')
print(f'  Precision   : {test_nano.box.mp:.4f}')
print(f'  Recall      : {test_nano.box.mr:.4f}')

Ultralytics 8.4.27 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M4)
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 1327.4±261.6 MB/s, size: 497.2 KB)
val: Scanning /Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/labels/test... 67 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 67/67 1.7Kit/s 0.0s
val: New cache created: /Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.3it/s 4.0s0.6s
                   all         67         72      0.942      0.897      0.936      0.561
Speed: 0.4ms preprocess, 34.4ms inference, 0.0ms loss, 6.4ms postprocess per image
Results saved to /Users/alijaafar/Documents/MLP/license-plate-detection/notebooks/runs/detect/val2

YOLOv8n — Test Split Metrics
  mAP@50      : 0.9364
  mAP@50:95   : 0.5606
  Precision   : 0.9417
  Recall      : 0.8971


### 5.1 Training Curves

In [9]:
def plot_training_curves(run_dir, model_name):
    """Plot loss and mAP curves from ultralytics results.csv."""
    csv_path = Path(run_dir) / 'results.csv'
    if not csv_path.exists():
        print(f'results.csv not found at {csv_path}')
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()   # ultralytics adds spaces

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=15)

    def safe_plot(ax, col, title, color='steelblue'):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=1.5)
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'Column not found:\n{col}',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)

    safe_plot(axes[0, 0], 'train/box_loss',  'Train Box Loss',  'tomato')
    safe_plot(axes[0, 1], 'train/cls_loss',  'Train Class Loss','darkorange')
    safe_plot(axes[0, 2], 'train/dfl_loss',  'Train DFL Loss',  'goldenrod')
    safe_plot(axes[1, 0], 'metrics/mAP50(B)',    'Val mAP@50',      'steelblue')
    safe_plot(axes[1, 1], 'metrics/mAP50-95(B)', 'Val mAP@50:95',   'mediumpurple')
    safe_plot(axes[1, 2], 'val/box_loss',    'Val Box Loss',    'seagreen')

    plt.tight_layout()
    save_path = RESULTS_DIR / f'{model_name}_training_curves.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


plot_training_curves(RESULTS_DIR / 'yolov8n', 'yolov8n')

<Figure size 1800x1000 with 6 Axes>

Saved: /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo/yolov8n_training_curves.png


### 5.2 Built-in Ultralytics Plots

Ultralytics automatically saves these plots to the run directory:

In [10]:
def show_run_plots(run_dir, model_name, plot_names=None):
    """Display PNG plots saved by ultralytics in the run directory."""
    if plot_names is None:
        plot_names = ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'results.png']

    found = []
    for name in plot_names:
        p = Path(run_dir) / name
        if p.exists():
            found.append(p)

    if not found:
        print('No plots found yet — run training first.')
        return

    fig, axes = plt.subplots(1, len(found), figsize=(7 * len(found), 6))
    if len(found) == 1:
        axes = [axes]

    for ax, p in zip(axes, found):
        img = np.array(Image.open(p))
        ax.imshow(img)
        ax.set_title(p.name, fontsize=11)
        ax.axis('off')

    fig.suptitle(f'{model_name} — Evaluation Plots', fontsize=14)
    plt.tight_layout()
    plt.show()


show_run_plots(RESULTS_DIR / 'yolov8n', 'yolov8n')

<Figure size 1400x600 with 2 Axes>

## 6. Visualise Predictions on Test Images

In [11]:
def predict_and_plot(model, img_paths, model_name, conf=0.25, max_imgs=9):
    """Run inference and display predictions with ground-truth boxes side by side."""
    sample = random.sample(img_paths, min(max_imgs, len(img_paths)))
    n_cols = 3
    n_rows = (len(sample) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, sample):
        img_path = Path(img_path)

        # Ground-truth label
        lbl_path = (
            img_path.parent.parent.parent   # yolo/
            / 'labels' / 'test'
            / img_path.with_suffix('.txt').name
        )

        img = np.array(Image.open(img_path).convert('RGB'))
        h, w = img.shape[:2]
        ax.imshow(img)

        # Draw ground-truth (green)
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    _, cx, cy, bw, bh = map(float, line.strip().split())
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    ax.add_patch(patches.Rectangle(
                        (x1, y1), bw * w, bh * h,
                        linewidth=2, edgecolor='lime', facecolor='none',
                        label='GT'
                    ))

        # Draw predictions (red)
        results = model.predict(str(img_path), conf=conf, device=DEVICE, verbose=False)
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf_score = box.conf[0].item()
                ax.add_patch(patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor='red', facecolor='none'
                ))
                ax.text(x1, y1 - 6, f'{conf_score:.2f}',
                        color='red', fontsize=8, fontweight='bold',
                        backgroundcolor='black')

        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    # Hide unused axes
    for ax in axes[len(sample):]:
        ax.axis('off')

    # Legend
    from matplotlib.lines import Line2D
    legend = [
        Line2D([0], [0], color='lime', linewidth=2, label='Ground Truth'),
        Line2D([0], [0], color='red',  linewidth=2, label=f'Prediction (conf>{conf})')
    ]
    fig.legend(handles=legend, loc='lower center', ncol=2, fontsize=11, framealpha=0.9)
    fig.suptitle(f'{model_name} — Predictions on Test Images', fontsize=14, y=1.01)
    plt.tight_layout()

    save_path = RESULTS_DIR / f'{model_name}_test_predictions.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


test_images = sorted(glob.glob(str(TEST_IMG_DIR / '*')))
print(f'Test images found: {len(test_images)}')

random.seed(42)
predict_and_plot(model_nano_best, test_images, 'yolov8n')

Test images found: 67


<Figure size 1800x1800 with 9 Axes>

Saved: /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo/yolov8n_test_predictions.png


## 7. Train YOLOv8s (Small)

YOLOv8s has ~3× more parameters than nano — usually gives a notable mAP boost.

In [12]:
model_small = YOLO('yolov8s.pt')
print(model_small.info())

YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs
(129, 11166560, 0, 28.816844800000002)


In [13]:
results_small = model_small.train(
    data      = str(DATA_YAML),
    epochs    = EPOCHS,
    batch     = BATCH,
    imgsz     = IMG_SIZE,
    device    = DEVICE,
    patience  = PATIENCE,
    project   = str(RESULTS_DIR),
    name      = 'yolov8s',
    exist_ok  = True,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.0,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.0,
    plots     = True,
    save      = True,
    verbose   = True,
)

print('\nYOLOv8s training complete.')
print(f'Results saved to: {RESULTS_DIR / "yolov8s"}')

Ultralytics 8.4.27 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

## 8. Validate YOLOv8s — Metrics

In [14]:
best_small = RESULTS_DIR / 'yolov8s' / 'weights' / 'best.pt'
model_small_best = YOLO(str(best_small))

val_small = model_small_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'val',
)

test_small = model_small_best.val(
    data   = str(DATA_YAML),
    imgsz  = IMG_SIZE,
    device = DEVICE,
    split  = 'test',
)

print('\n' + '=' * 50)
print('YOLOv8s — Test Split Metrics')
print('=' * 50)
print(f'  mAP@50      : {test_small.box.map50:.4f}')
print(f'  mAP@50:95   : {test_small.box.map:.4f}')
print(f'  Precision   : {test_small.box.mp:.4f}')
print(f'  Recall      : {test_small.box.mr:.4f}')

Ultralytics 8.4.27 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M4)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1378.9±463.0 MB/s, size: 498.4 KB)
val: Scanning /Users/alijaafar/Documents/MLP/license-plate-detection/data/processed/yolo/labels/val.cache... 64 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 33.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.8s/it 15.2s5.7ss
                   all         64         67      0.934      0.925      0.938      0.606
Speed: 0.3ms preprocess, 209.7ms inference, 0.0ms loss, 5.3ms postprocess per image
Results saved to /Users/alijaafar/Documents/MLP/license-plate-detection/notebooks/runs/detect/val3
Ultralytics 8.4.27 🚀 Python-3.9.6 torch-2.8.0 MPS (Apple M4)
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 1014.4±394.3 MB/s, size: 407.4 KB)
val: Scanning /Users/alijaafar/Documen

In [15]:
plot_training_curves(RESULTS_DIR / 'yolov8s', 'yolov8s')
show_run_plots(RESULTS_DIR / 'yolov8s', 'yolov8s')

<Figure size 1800x1000 with 6 Axes>

Saved: /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo/yolov8s_training_curves.png


<Figure size 1400x600 with 2 Axes>

In [16]:
random.seed(42)
predict_and_plot(model_small_best, test_images, 'yolov8s')

<Figure size 1800x1800 with 9 Axes>

Saved: /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo/yolov8s_test_predictions.png


## 9. Compare YOLOv8n vs YOLOv8s

In [17]:
metrics = {
    'Model':      ['YOLOv8n', 'YOLOv8s'],
    'mAP@50':     [test_nano.box.map50,  test_small.box.map50],
    'mAP@50:95':  [test_nano.box.map,    test_small.box.map],
    'Precision':  [test_nano.box.mp,     test_small.box.mp],
    'Recall':     [test_nano.box.mr,     test_small.box.mr],
    'Weights (MB)': [
        round(os.path.getsize(best_nano) / 1e6, 1),
        round(os.path.getsize(best_small) / 1e6, 1),
    ],
}

df_compare = pd.DataFrame(metrics)
print('=' * 60)
print('YOLOv8 Model Comparison — Test Split')
print('=' * 60)
print(df_compare.to_string(index=False, float_format='{:.4f}'.format))

# Save to CSV for use in evaluation notebook
save_path = RESULTS_DIR / 'yolov8_comparison.csv'
df_compare.to_csv(save_path, index=False)
print(f'\nSaved: {save_path}')

YOLOv8 Model Comparison — Test Split
  Model  mAP@50  mAP@50:95  Precision  Recall  Weights (MB)
YOLOv8n  0.9364     0.5606     0.9417  0.8971        6.3000
YOLOv8s  0.9011     0.5315     0.8884  0.8611       22.5000

Saved: /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo/yolov8_comparison.csv


In [18]:
# Bar chart comparison
metrics_cols = ['mAP@50', 'mAP@50:95', 'Precision', 'Recall']
x  = np.arange(len(metrics_cols))
bw = 0.3

fig, ax = plt.subplots(figsize=(11, 5))
bars_n = ax.bar(x - bw / 2, df_compare.loc[0, metrics_cols], bw,
                label='YOLOv8n', color='steelblue', alpha=0.85)
bars_s = ax.bar(x + bw / 2, df_compare.loc[1, metrics_cols], bw,
                label='YOLOv8s', color='coral',     alpha=0.85)

for bars in [bars_n, bars_s]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics_cols, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('YOLOv8n vs YOLOv8s — Test Split Metrics', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

save_path = RESULTS_DIR / 'yolov8_comparison_chart.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

<Figure size 1100x500 with 1 Axes>

Saved: /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo/yolov8_comparison_chart.png


## 10. Save Best Weights to `models/`

In [19]:
# Copy best.pt and last.pt for both models into models/
def save_model_weights(run_dir, model_name, models_dir):
    run_dir    = Path(run_dir)
    models_dir = Path(models_dir)
    weights    = ['best.pt', 'last.pt']
    for w in weights:
        src = run_dir / 'weights' / w
        if src.exists():
            dst = models_dir / f'{model_name}_{w}'
            shutil.copy2(src, dst)
            size_mb = dst.stat().st_size / 1e6
            print(f'  Saved {dst.name}  ({size_mb:.1f} MB)')
        else:
            print(f'  WARNING: {src} not found')


print('Saving YOLOv8n weights...')
save_model_weights(RESULTS_DIR / 'yolov8n', 'yolov8n', MODELS_DIR)

print('Saving YOLOv8s weights...')
save_model_weights(RESULTS_DIR / 'yolov8s', 'yolov8s', MODELS_DIR)

print('\nAll weights saved to:', MODELS_DIR)
for f in sorted(MODELS_DIR.iterdir()):
    if f.suffix == '.pt':
        print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

Saving YOLOv8n weights...
  Saved yolov8n_best.pt  (6.3 MB)
  Saved yolov8n_last.pt  (6.3 MB)
Saving YOLOv8s weights...
  Saved yolov8s_best.pt  (22.5 MB)
  Saved yolov8s_last.pt  (22.5 MB)

All weights saved to: /Users/alijaafar/Documents/MLP/license-plate-detection/models
  yolov8n_best.pt  (6.3 MB)
  yolov8n_last.pt  (6.3 MB)
  yolov8s_best.pt  (22.5 MB)
  yolov8s_last.pt  (22.5 MB)


## 11. Final Summary

In [20]:
print('=' * 60)
print('YOLOV8 TRAINING SUMMARY')
print('=' * 60)
print(f'Device       : {DEVICE}')
print(f'Epochs       : {EPOCHS} (patience={PATIENCE})')
print(f'Batch size   : {BATCH}')
print(f'Image size   : {IMG_SIZE}')
print()
print(df_compare.to_string(index=False, float_format='{:.4f}'.format))
print()

best_model = df_compare.loc[df_compare['mAP@50'].idxmax(), 'Model']
best_map50 = df_compare['mAP@50'].max()
print(f'Best model   : {best_model}  (mAP@50 = {best_map50:.4f})')
print()
print('Weights saved in:', MODELS_DIR)
print('Plots saved in  :', RESULTS_DIR)
print()
print('=' * 60)
print('NEXT: 04_train_faster_rcnn.ipynb')
print('=' * 60)

YOLOV8 TRAINING SUMMARY
Device       : mps
Epochs       : 100 (patience=20)
Batch size   : 16
Image size   : 640

  Model  mAP@50  mAP@50:95  Precision  Recall  Weights (MB)
YOLOv8n  0.9364     0.5606     0.9417  0.8971        6.3000
YOLOv8s  0.9011     0.5315     0.8884  0.8611       22.5000

Best model   : YOLOv8n  (mAP@50 = 0.9364)

Weights saved in: /Users/alijaafar/Documents/MLP/license-plate-detection/models
Plots saved in  : /Users/alijaafar/Documents/MLP/license-plate-detection/results/yolo

NEXT: 04_train_faster_rcnn.ipynb


---
## Next Steps

| Notebook | Content |
|---|---|
| `04_train_faster_rcnn.ipynb` | Train Faster R-CNN (ResNet-50 FPN) using COCO JSON annotations |
| `05_train_retinanet.ipynb` | Train RetinaNet using COCO JSON annotations |
| `06_evaluation.ipynb` | Final comparison: mAP, precision, recall, FPS across all models |

### Colab tip
Set `DEVICE = '0'` when running on Colab with a T4/A100 GPU.